# MIST Glioma demo

Go from raw DICOM images to a glioma segmentation on a free GPU. This notebook uses public case UPENN-GBM-00622 and the published BraTS 2025 weights.

Choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

The example data is from the [UPENN-GBM collection](https://doi.org/10.7937/TCIA.709X-DN49) (CC BY 4.0, citation required). By running this notebook you agree to the TCIA Data Usage Policy. **The output is for research use only and is not for clinical use.**

## 1. Install

This takes a few minutes and uses its own environment, so Colab's packages stay untouched.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content")
VENV = ROOT / "venv"
PYTHON = str(VENV / "bin" / "python")
CASE = ROOT / "data" / "UPENN-GBM-00622"
OUTPUT = ROOT / "output"
PACKAGE = "mist-glioma"
DOWNLOAD_SCRIPT = "https://raw.githubusercontent.com/mist-medical/mist-glioma/main/examples/download_upenn_gbm_00622.py"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "venv", str(VENV), "--python", "3.12", "--clear"], check=True)
subprocess.run(["uv", "pip", "install", "-q", "--python", PYTHON, PACKAGE], check=True)

ENV = {key: value for key, value in os.environ.items() if key not in ("MPLBACKEND", "PYTHONPATH")}
ENV["PATH"] = f"{VENV / 'bin'}:{ENV['PATH']}"
gpu = subprocess.run([PYTHON, "-c", "import torch; print(torch.cuda.is_available())"], env=ENV, capture_output=True, text=True, check=True).stdout
assert "True" in gpu, "No GPU found. Choose Runtime > Change runtime type > GPU."
print("Ready")

## 2. Download the example case

This is about 25 MB of DICOM images: T1, T2, contrast-enhanced T1, and FLAIR.

In [ ]:
subprocess.run(["curl", "-fsSL", "-o", str(ROOT / "download_example.py"), DOWNLOAD_SCRIPT], check=True)
subprocess.run([PYTHON, str(ROOT / "download_example.py"), "--output", str(CASE)], env=ENV, check=True)

## 3. Segment

This one command handles registration, brain extraction, and inference.

In [ ]:
subprocess.run(
    [
        "mist_glioma",
        "--t1", str(CASE / "t1"),
        "--t2", str(CASE / "t2"),
        "--tc", str(CASE / "tc"),
        "--fl", str(CASE / "fl"),
        "--output", str(OUTPUT),
        "--device", "cuda",
    ],
    env=ENV,
    check=True,
)
print(sorted(path.name for path in OUTPUT.iterdir()))

## 4. View the result

Yellow is non-enhancing tumor core, green is surrounding FLAIR hyperintensity, red is enhancing tissue, and blue is resection cavity. FLAIR is resampled onto the contrast-enhanced T1 grid for display.

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from matplotlib.colors import ListedColormap
from nibabel.processing import resample_from_to
from scipy import ndimage

subprocess.run(
    [PYTHON, "-c", f'''
import ants
from mist_glioma.image_io import read_image
for name in ("tc", "fl"):
    ants.image_write(read_image("{CASE}/" + name), "{ROOT}/" + name + ".nii.gz")
'''],
    env=ENV,
    cwd=ROOT,
    check=True,
)
tc_image = nib.as_closest_canonical(nib.load(ROOT / "tc.nii.gz"))
fl_image = resample_from_to(nib.load(ROOT / "fl.nii.gz"), tc_image, order=1)
mask_image = nib.as_closest_canonical(nib.load(OUTPUT / "tumor_rc_mask.nii.gz"))
tc = np.asanyarray(tc_image.dataobj)
fl = np.asanyarray(fl_image.dataobj)
mask = np.asanyarray(mask_image.dataobj)

COLORS = ListedColormap([(0, 0, 0, 0), "#ffd400", "#2ecc40", "#ff2d2d", "#2f7fff"])
center = [int(np.rint(value)) for value in ndimage.center_of_mass(mask > 0)]
zooms = tc_image.header.get_zooms()


def take(volume, axis):
    return np.rot90(np.take(volume, center[axis], axis=axis))


figure, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, (axis, aspect) in enumerate(((2, zooms[1] / zooms[0]), (1, zooms[2] / zooms[0]))):
    for column, (volume, title, overlay) in enumerate(
        ((tc, "T1 post-contrast", False), (tc, "T1 post-contrast + mask", True), (fl, "FLAIR", False), (fl, "FLAIR + mask", True))
    ):
        axes[row, column].imshow(take(volume, axis), cmap="gray", aspect=aspect, vmin=0, vmax=np.percentile(volume[volume > 0], 99.5))
        if overlay:
            axes[row, column].imshow(take(mask, axis), cmap=COLORS, vmin=0, vmax=4, alpha=0.5, aspect=aspect, interpolation="nearest")
        axes[row, column].set_title(title)
        axes[row, column].axis("off")
figure.tight_layout()
plt.show()

voxel_cc = float(np.prod(zooms[:3])) / 1000
regions = {"Non-enhancing tumor core": [1], "FLAIR hyperintensity": [2], "Enhancing tissue": [3], "Resection cavity": [4]}
for name, labels in regions.items():
    print(f"{name}: {np.isin(mask, labels).sum() * voxel_cc:.1f} cc")